# LeetCode 1292 — Maximum Side Length of a Square with Sum Less than or Equal to Threshold bằng Python

Link đề: https://leetcode.com/problems/maximum-side-length-of-a-square-with-sum-less-than-or-equal-to-threshold/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Matrix, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def maxSideLength(self, mat: list[list[int]], threshold: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Thành thạo **prefix sum 2D**: dựng bảng `P` và tính tổng một hình vuông bất kỳ trong `O(1)`.
2. Nhận ra **tính đơn điệu** của đáp án (vì `mat[i][j] ≥ 0`): nếu có hình vuông cạnh `k` thoả thì cũng có hình vuông cạnh `k−1` thoả ⇒ **binary search trên cạnh**.
3. Hiểu mẹo **tăng dần đáp án** `ans → ans+1`: chỉ thử đúng một cạnh tại mỗi ô ⇒ tổng `O(mn)`.
4. Đi từ brute force `O(mn·L³)` → `O(mn·L)` → `O(mn log L)` → `O(mn)`.
5. Tránh lỗi: sai dấu công thức prefix, binary search "tìm lớn nhất" bị lặp vô hạn, chỉ số âm trong Python **không báo lỗi** mà lặng lẽ cho kết quả sai.

## 2. Đề bài & ví dụ

Cho ma trận `mat` kích thước `m × n` (số nguyên không âm) và số nguyên `threshold`. Trả về **độ dài cạnh lớn nhất** của một hình vuông con có **tổng các phần tử ≤ `threshold`**; nếu không có hình vuông nào, trả về `0`.

**Ví dụ 1:**
```
mat = [[1,1,3,2,4,3,2],
       [1,1,3,2,4,3,2],
       [1,1,3,2,4,3,2]], threshold = 4
```
→ `2` (hình vuông 2×2 toàn số 1 ở góc trái có tổng 4).

**Ví dụ 2:** `mat` là ma trận 5×5 toàn số 2, `threshold = 1` → `0` (mọi ô đều > 1).

**Ràng buộc:** `1 <= m, n <= 300`, `0 <= mat[i][j] <= 10^4`, `0 <= threshold <= 10^5`.

In [1]:
import random
import time
from typing import List

EX1 = [[1, 1, 3, 2, 4, 3, 2],
       [1, 1, 3, 2, 4, 3, 2],
       [1, 1, 3, 2, 4, 3, 2]]
EX2 = [[2] * 5 for _ in range(5)]
examples = [(EX1, 4, 2), (EX2, 1, 0)]
for mat, th, want in examples:
    print(f"{len(mat)}x{len(mat[0])}, threshold={th} -> mong đợi {want}")

3x7, threshold=4 -> mong đợi 2
5x5, threshold=1 -> mong đợi 0


## 3. Kiến thức nền

### 3.1 Prefix sum 1D (nhắc lại)

`pre[i]` = tổng `a[0..i-1]`, `pre[0] = 0`. Tổng đoạn `a[l..r]` = `pre[r+1] − pre[l]`.

In [2]:
a = [3, 1, 4, 1, 5]
pre = [0]
for x in a:
    pre.append(pre[-1] + x)
print("pre =", pre, "| tổng a[1..3] =", pre[4] - pre[1])

pre = [0, 3, 4, 8, 9, 14] | tổng a[1..3] = 6


### 3.2 Prefix sum 2D

`P[i][j]` = tổng hình chữ nhật từ `(0,0)` đến `(i−1, j−1)` (bảng lệch 1, có viền 0):

```
P[i][j] = mat[i-1][j-1] + P[i-1][j] + P[i][j-1] − P[i-1][j-1]
```

Trừ `P[i−1][j−1]` vì phần đó bị cộng **hai lần**.

In [3]:
def build_prefix(mat):
    m, n = len(mat), len(mat[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            P[i][j] = mat[i - 1][j - 1] + P[i - 1][j] + P[i][j - 1] - P[i - 1][j - 1]
    return P

small = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
for row in build_prefix(small):
    print(row)

[0, 0, 0, 0]
[0, 1, 3, 6]
[0, 5, 12, 21]
[0, 12, 27, 45]


### 3.3 Tổng hình vuông cạnh `k` có góc dưới-phải tại `(i, j)` (chỉ số của `P`)

Hình vuông gồm các hàng `i−k … i−1`, cột `j−k … j−1` của `mat`:

```
sum = P[i][j] − P[i-k][j] − P[i][j-k] + P[i-k][j-k]
```

Cần `i ≥ k` và `j ≥ k`.

In [4]:
def square_sum(P, i, j, k):
    return P[i][j] - P[i - k][j] - P[i][j - k] + P[i - k][j - k]

P = build_prefix(small)
print("hình vuông 2x2 góc dưới-phải (2,2) của mat:", square_sum(P, 3, 3, 2), "= 5+6+8+9")
print("toàn bộ 3x3:", square_sum(P, 3, 3, 3))

hình vuông 2x2 góc dưới-phải (2,2) của mat: 28 = 5+6+8+9
toàn bộ 3x3: 45


### 3.4 Binary search "tìm giá trị lớn nhất thoả"

Vị từ dạng `True, …, True, False, …, False`. Mẫu:

```python
lo, hi = 0, L          # ok(0) luôn đúng
while lo < hi:
    mid = (lo + hi + 1) // 2      # làm tròn LÊN
    if ok(mid): lo = mid
    else:       hi = mid - 1
return lo
```

Làm tròn lên để `mid > lo`, tránh kẹt khi `hi = lo + 1`.

In [5]:
def last_true(lo, hi, ok):
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

print(last_true(0, 100, lambda x: x * x <= 50))   # 7

7


### 3.5 Cẩn thận: chỉ số âm trong Python

`a[-1]` là phần tử **cuối**, không phải lỗi! Nếu công thức vô tình dùng `P[i - k]` với `i < k`, Python không báo lỗi mà đọc một hàng ở cuối bảng ⇒ sai lặng lẽ. Luôn kiểm tra `i ≥ k`, `j ≥ k`.

In [6]:
row = [10, 20, 30]
print("row[-1] =", row[-1], "| row[0 - 2] =", row[0 - 2], "-> không có lỗi nào được báo!")

row[-1] = 30 | row[0 - 2] = 20 -> không có lỗi nào được báo!


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1** (`threshold = 4`):

- Cạnh 1: ô nhỏ nhất là 1 ≤ 4 ⇒ có.
- Cạnh 2: hình vuông 2×2 ở cột 0–1: `1+1+1+1 = 4 ≤ 4` ⇒ có.
- Cạnh 3: hình vuông 3×3 nhỏ nhất (cột 0–2): `3·(1+1+3) = 15 > 4` ⇒ không có.
- Đáp án `2`.

**Tính đơn điệu:** vì mọi phần tử `≥ 0`, nếu một hình vuông cạnh `k` có tổng `≤ threshold` thì hình vuông cạnh `k−1` nằm **bên trong** nó có tổng **không lớn hơn** ⇒ cũng thoả. Vậy tập các cạnh "có thể" là `{0, 1, …, ans}` — một đoạn liên tục. Đây là chìa khoá cho binary search và cho mẹo tăng dần.

In [7]:
P1 = build_prefix(EX1)
m, n = len(EX1), len(EX1[0])
for k in range(1, min(m, n) + 1):
    best = min(square_sum(P1, i, j, k) for i in range(k, m + 1) for j in range(k, n + 1))
    print(f"cạnh {k}: tổng nhỏ nhất = {best} -> {'thoả' if best <= 4 else 'không thoả'}")

cạnh 1: tổng nhỏ nhất = 1 -> thoả
cạnh 2: tổng nhỏ nhất = 4 -> thoả
cạnh 3: tổng nhỏ nhất = 15 -> không thoả


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** thử mọi góc trên-trái `(r, c)` và mọi cạnh `k`; tính tổng bằng hai vòng lặp. Nhờ tính đơn điệu, với mỗi `(r, c)` có thể dừng khi tổng vượt ngưỡng.

In [8]:
def max_side_brute(mat, threshold):
    m, n = len(mat), len(mat[0])
    best = 0
    for r in range(m):
        for c in range(n):
            for k in range(1, min(m - r, n - c) + 1):
                s = sum(mat[i][j] for i in range(r, r + k) for j in range(c, c + k))
                if s > threshold:
                    break                      # cạnh lớn hơn còn tổng lớn hơn
                best = max(best, k)
    return best

for mat, th, want in examples:
    print(max_side_brute(mat, th), "| mong đợi", want)

2 | mong đợi 2
0 | mong đợi 0


**Độ phức tạp:** worst `O(mn · L · k²) = O(mn·L³)` với `L = min(m, n)` (ví dụ ma trận toàn 0, `threshold` lớn — không bao giờ `break`). Với `300 × 300` là ~`10^12` phép.

**Nút thắt:** tính lại tổng hình vuông từ đầu mỗi lần. Prefix sum 2D đưa việc này về `O(1)`.

In [9]:
for size in [10, 20, 30]:
    z = [[0] * size for _ in range(size)]
    t = time.perf_counter()
    max_side_brute(z, 0)
    print(f"{size}x{size} toàn 0: brute force {time.perf_counter() - t:.3f}s")

10x10 toàn 0: brute force 0.000s
20x20 toàn 0: brute force 0.006s
30x30 toàn 0: brute force 0.038s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — prefix sum:** tổng hình vuông bất kỳ trong `O(1)`.

**Insight 2 — đơn điệu:** `exists(k)` := "có hình vuông cạnh `k` với tổng `≤ threshold`" có dạng `True…True False…False` ⇒ binary search `k`; mỗi lần kiểm tra `exists(k)` tốn `O(mn)`.

**Insight 3 — tăng dần đáp án:** duyệt các ô `(i, j)` làm góc dưới-phải. Giữ `ans` = cạnh tốt nhất đã thấy. Tại mỗi ô, **chỉ cần thử cạnh `ans + 1`**:
- Nếu `i ≥ ans+1`, `j ≥ ans+1` và tổng hình vuông cạnh `ans+1` `≤ threshold` ⇒ `ans += 1`.
- Không cần thử cạnh `≤ ans` (không cải thiện), cũng không cần thử `ans + 2` ngay (xem chứng minh ở mục 10).

**Quy tắc "nếu… thì…":**
- Nếu ô nhỏ nhất `> threshold` ⇒ đáp án 0.
- Nếu `exists(k)` sai ⇒ mọi `k' > k` cũng sai.
- Nếu tại ô `(i, j)` cạnh `ans+1` thoả ⇒ tăng `ans` đúng 1.

## 7. Các cách tiếp theo

### Cách 2 — Prefix sum 2D + thử mọi hình vuông

**Từng đoạn:**
1. Dựng `P`.
2. Với mỗi góc dưới-phải `(i, j)` (chỉ số `P`), thử `k = 1 … min(i, j)`, dừng khi vượt ngưỡng.

In [10]:
def max_side_prefix(mat, threshold):
    m, n = len(mat), len(mat[0])
    P = build_prefix(mat)
    best = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            for k in range(1, min(i, j) + 1):
                if square_sum(P, i, j, k) > threshold:
                    break
                best = max(best, k)
    return best

for mat, th, want in examples:
    print(max_side_prefix(mat, th), "| mong đợi", want)

2 | mong đợi 2
0 | mong đợi 0


**Độ phức tạp Cách 2:** `O(mn·L)` worst (≈ `2.7·10^7` với 300×300 toàn 0 — chậm trong Python), `O(mn)` bộ nhớ.

Mẹo nhỏ: chỉ cần thử `k` **lớn hơn `best`** — bắt đầu từ `k = best + 1` thay vì 1. Đây chính là mầm mống của Cách 4.

### Cách 3 — Prefix sum + binary search trên cạnh

**Từng đoạn:**
1. `exists(k)`: duyệt mọi góc dưới-phải `(i, j)` với `i, j ≥ k`, trả `True` nếu có tổng `≤ threshold`.
2. `last_true(0, L, exists)` — `exists(0)` coi như luôn đúng.

In [11]:
def max_side_binary(mat, threshold):
    m, n = len(mat), len(mat[0])
    P = build_prefix(mat)

    def exists(k):
        if k == 0:
            return True
        for i in range(k, m + 1):
            for j in range(k, n + 1):
                if P[i][j] - P[i - k][j] - P[i][j - k] + P[i - k][j - k] <= threshold:
                    return True
        return False

    lo, hi = 0, min(m, n)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if exists(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

for mat, th, want in examples:
    print(max_side_binary(mat, th), "| mong đợi", want)

2 | mong đợi 2
0 | mong đợi 0


In [12]:
class SolutionBinary:
    def maxSideLength(self, mat: List[List[int]], threshold: int) -> int:
        m, n = len(mat), len(mat[0])
        P = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(m):
            row_sum = 0
            for j in range(n):
                row_sum += mat[i][j]
                P[i + 1][j + 1] = P[i][j + 1] + row_sum      # cách dựng khác: cộng dồn theo hàng

        def exists(k):
            for i in range(k, m + 1):
                for j in range(k, n + 1):
                    if P[i][j] - P[i - k][j] - P[i][j - k] + P[i - k][j - k] <= threshold:
                        return True
            return False

        lo, hi = 0, min(m, n)
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if exists(mid):
                lo = mid
            else:
                hi = mid - 1
        return lo

print([SolutionBinary().maxSideLength(mat, th) for mat, th, _ in examples])

[2, 0]


`SolutionBinary` dựng `P` bằng cách khác: `P[i+1][j+1] = P[i][j+1] + (tổng hàng i từ cột 0 tới j)` — tương đương công thức bao hàm–loại trừ nhưng chỉ có một phép cộng.

**Độ phức tạp Cách 3:** `O(mn)` dựng + `O(mn · log L)` tìm kiếm, `O(mn)` bộ nhớ.

### Cách 4 — Tăng dần đáp án `ans → ans + 1`, tổng `O(mn)`

**Từng đoạn:**
1. Dựng `P` (có thể dựng **đồng thời** khi duyệt vì chỉ cần các ô đã qua).
2. Duyệt `(i, j)` theo hàng; nếu `i > ans` và `j > ans` (tức `≥ ans+1`), thử cạnh `k = ans + 1`.
3. Thoả ⇒ `ans += 1`.

In [13]:
def max_side_incremental(mat, threshold):
    m, n = len(mat), len(mat[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]
    ans = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            P[i][j] = mat[i - 1][j - 1] + P[i - 1][j] + P[i][j - 1] - P[i - 1][j - 1]
            k = ans + 1
            if i >= k and j >= k and P[i][j] - P[i - k][j] - P[i][j - k] + P[i - k][j - k] <= threshold:
                ans = k
    return ans

for mat, th, want in examples:
    print(max_side_incremental(mat, th), "| mong đợi", want)

2 | mong đợi 2
0 | mong đợi 0


**Độ phức tạp Cách 4:** mỗi ô `O(1)` ⇒ `O(mn)` thời gian, `O(mn)` bộ nhớ.

### Cách 5 — Binary search cạnh lớn nhất **tại từng ô** (tham khảo)

Với mỗi góc dưới-phải, cạnh thoả cũng đơn điệu ⇒ binary search riêng cho từng ô: `O(mn log L)`. Không tốt hơn Cách 4, nhưng là bước đệm để giải biến thể **đếm** số hình vuông thoả (Bài tập 5).

In [14]:
def max_side_per_cell(mat, threshold):
    m, n = len(mat), len(mat[0])
    P = build_prefix(mat)
    best = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            k = last_true(0, min(i, j), lambda k: k == 0 or square_sum(P, i, j, k) <= threshold)
            best = max(best, k)
    return best

for mat, th, want in examples:
    print(max_side_per_cell(mat, th), "| mong đợi", want)

2 | mong đợi 2
0 | mong đợi 0


## 8. TRACE từng bước

Trace Cách 4 trên **Ví dụ 1** (chỉ in những ô có thử cạnh `ans+1`). Dòng đầu mọi ô có `i = 1`: chỉ thử cạnh 1.

| (i, j) | ans trước | thử k | tổng | ≤ 4? | ans sau |
|---|---|---|---|---|---|
| (1,1) | 0 | 1 | 1 | ✔ | 1 |
| (1,2…7) | 1 | 2 | — (i < 2) | — | 1 |
| (2,1) | 1 | 2 | — (j < 2) | — | 1 |
| (2,2) | 1 | 2 | 1+1+1+1 = 4 | ✔ | 2 |
| (2,3…7) | 2 | 3 | — (i < 3) | — | 2 |
| (3,3) | 2 | 3 | 15 | ✗ | 2 |
| (3,4…7) | 2 | 3 | 24, 33, 30, 27 | ✗ | 2 |

In [15]:
def trace_incremental(mat, threshold):
    m, n = len(mat), len(mat[0])
    P = [[0] * (n + 1) for _ in range(m + 1)]
    ans = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            P[i][j] = mat[i - 1][j - 1] + P[i - 1][j] + P[i][j - 1] - P[i - 1][j - 1]
            k = ans + 1
            if i >= k and j >= k:
                s = P[i][j] - P[i - k][j] - P[i][j - k] + P[i - k][j - k]
                ok = s <= threshold
                if ok:
                    ans = k
                print(f"(i={i}, j={j}): thử cạnh {k}, tổng={s:>2} {'<=' if ok else '> '} {threshold}"
                      f" -> ans={ans}")
    return ans

print("đáp án =", trace_incremental(EX1, 4))

(i=1, j=1): thử cạnh 1, tổng= 1 <= 4 -> ans=1
(i=2, j=2): thử cạnh 2, tổng= 4 <= 4 -> ans=2
(i=3, j=3): thử cạnh 3, tổng=15 >  4 -> ans=2
(i=3, j=4): thử cạnh 3, tổng=18 >  4 -> ans=2
(i=3, j=5): thử cạnh 3, tổng=27 >  4 -> ans=2
(i=3, j=6): thử cạnh 3, tổng=27 >  4 -> ans=2
(i=3, j=7): thử cạnh 3, tổng=27 >  4 -> ans=2
đáp án = 2


In [16]:
def trace_binary(mat, threshold):
    m, n = len(mat), len(mat[0])
    P = build_prefix(mat)
    def exists(k):
        return any(square_sum(P, i, j, k) <= threshold
                   for i in range(k, m + 1) for j in range(k, n + 1))
    lo, hi = 0, min(m, n)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        e = exists(mid)
        print(f"lo={lo}, hi={hi}, mid={mid}: exists({mid}) = {e} -> "
              + (f"lo = {mid}" if e else f"hi = {mid - 1}"))
        if e:
            lo = mid
        else:
            hi = mid - 1
    print("đáp án =", lo)

trace_binary(EX1, 4)
trace_binary([[0] * 6 for _ in range(6)], 0)

lo=0, hi=3, mid=2: exists(2) = True -> lo = 2
lo=2, hi=3, mid=3: exists(3) = False -> hi = 2
đáp án = 2
lo=0, hi=6, mid=3: exists(3) = True -> lo = 3
lo=3, hi=6, mid=5: exists(5) = True -> lo = 5
lo=5, hi=6, mid=6: exists(6) = True -> lo = 6
đáp án = 6


## 9. Code nộp LeetCode

Chọn Cách 4 (tăng dần đáp án, dựng prefix đồng thời): `O(mn)`.

In [17]:
class Solution:
    def maxSideLength(self, mat: List[List[int]], threshold: int) -> int:
        m, n = len(mat), len(mat[0])
        P = [[0] * (n + 1) for _ in range(m + 1)]
        ans = 0
        for i in range(1, m + 1):
            row = mat[i - 1]
            for j in range(1, n + 1):
                P[i][j] = row[j - 1] + P[i - 1][j] + P[i][j - 1] - P[i - 1][j - 1]
                k = ans + 1                       # chỉ cần thử cạnh lớn hơn đáp án hiện tại 1
                if i >= k and j >= k:
                    s = P[i][j] - P[i - k][j] - P[i][j - k] + P[i - k][j - k]
                    if s <= threshold:
                        ans = k
        return ans

for mat, th, want in examples:
    got = Solution().maxSideLength(mat, th)
    print(got, "OK" if got == want else "SAI")

2 OK
0 OK


## 10. Vì sao thuật toán đúng

**Prefix sum:** `P[i][j]` là tổng hình chữ nhật `[0, i) × [0, j)`; tổng hình vuông theo bao hàm–loại trừ.

**Binary search (Cách 3):** `mat ≥ 0` ⇒ hình vuông cạnh `k−1` nằm trong hình vuông cạnh `k` có tổng `≤` ⇒ `exists(k) ⇒ exists(k−1)`. Vị từ đơn điệu ⇒ `last_true` trả đúng cạnh lớn nhất.

**Tăng dần (Cách 4):** gọi `A` là đáp án thật.
1. `ans` không bao giờ vượt `A`: mỗi lần tăng đều do tìm thấy một hình vuông cạnh `ans+1` thoả.
2. `ans` đạt `A`: gọi `(i*, j*)` là góc dưới-phải của một hình vuông cạnh `A` thoả. Với mọi `k ≤ A`, hình vuông cạnh `k` cùng góc `(i*, j*)` cũng thoả (nằm bên trong). Xét lúc duyệt tới `(i*, j*)`: nếu `ans < A` thì hình vuông cạnh `ans + 1` tại đây thoả ⇒ `ans` tăng 1. Nhưng chỉ tăng **1** — liệu có kịp đạt `A`? Có, theo lập luận mạnh hơn: với mỗi `t = 1..A`, hình vuông cạnh `t` có góc dưới-phải `(i* − (A − t), j* − (A − t))` (nằm ở góc trên-trái của hình vuông lớn) cũng thoả, và các ô này được duyệt **theo thứ tự trước** `(i*, j*)` theo đường chéo. Quy nạp theo `t`: khi tới ô của bước `t`, `ans ≥ t − 1` ⇒ sau ô đó `ans ≥ t`. Vậy sau `(i*, j*)`, `ans ≥ A`.

(Các ô trên đường chéo có cả `i` lẫn `j` tăng, nên ô có `t` nhỏ hơn luôn được duyệt trước theo thứ tự hàng.)

## 11. Phân tích Time / Space

Gọi `m, n` là kích thước, `L = min(m, n)`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(mn·L³)` worst | `O(1)` |
| 2. Prefix + thử mọi cạnh | `O(mn·L)` worst | `O(mn)` |
| 3. Prefix + binary search | `O(mn log L)` | `O(mn)` |
| 4. Tăng dần đáp án | `O(mn)` | `O(mn)` |
| 5. Binary search từng ô | `O(mn log L)` | `O(mn)` |

Worst case của Cách 1, 2 là ma trận toàn 0 với `threshold` lớn (không bao giờ `break`).

## 12. Bộ test

Ví dụ đề; edge case (1×1, `threshold = 0`, toàn 0, hàng/cột đơn, mọi ô > threshold, ngưỡng vừa khít); đối chiếu ngẫu nhiên với brute force (seed cố định, đúng ràng buộc); test hiệu năng `300 × 300`.

In [18]:
ALL = {
    "brute": max_side_brute, "prefix": max_side_prefix, "binary": max_side_binary,
    "incremental": max_side_incremental, "per_cell": max_side_per_cell,
    "SolBinary": lambda a, t: SolutionBinary().maxSideLength(a, t),
    "Solution": lambda a, t: Solution().maxSideLength(a, t),
}
fixed = [
    (EX1, 4, 2), (EX2, 1, 0),
    ([[5]], 4, 0), ([[5]], 5, 1), ([[0]], 0, 1),
    ([[0, 0], [0, 0]], 0, 2),
    ([[1, 2, 3, 4]], 10, 1),                       # một hàng: cạnh tối đa 1
    ([[1], [1], [1]], 0, 0),
    ([[1, 1, 1], [1, 1, 1], [1, 1, 1]], 8, 2),     # 9 > 8 nên không có 3x3
    ([[1, 1, 1], [1, 1, 1], [1, 1, 1]], 9, 3),     # vừa khít
    ([[10**4] * 3 for _ in range(3)], 10**5, 3),
    ([[18, 70], [61, 1], [25, 85], [14, 40], [11, 96], [97, 96], [63, 45]], 40184, 2),
]
for mat, th, want in fixed:
    for name, f in ALL.items():
        got = f(mat, th)
        assert got == want, (name, mat, th, got, want)
print("Test cố định: qua", len(fixed), "case")

Test cố định: qua 12 case


In [19]:
rng = random.Random(1292)
for _ in range(500):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    hi_v = rng.choice([0, 3, 10, 10**4])
    mat = [[rng.randint(0, hi_v) for _ in range(n)] for _ in range(m)]
    th = rng.randint(0, min(10**5, max(1, sum(map(sum, mat)))))
    want = max_side_brute(mat, th)
    for name, f in ALL.items():
        assert f(mat, th) == want, (name, mat, th)
print("500 ma trận ngẫu nhiên: mọi cách khớp brute force")

500 ma trận ngẫu nhiên: mọi cách khớp brute force


In [20]:
big_zero = [[0] * 300 for _ in range(300)]
big_rand = [[rng.randint(0, 10**4) for _ in range(300)] for _ in range(300)]
big_small = [[rng.randint(0, 3) for _ in range(300)] for _ in range(300)]
for name in ["incremental", "Solution", "binary", "SolBinary"]:
    t = time.perf_counter()
    r = (ALL[name](big_zero, 10**5), ALL[name](big_rand, 10**5), ALL[name](big_small, 500))
    print(f"{name:>11}: {r}, {time.perf_counter() - t:.3f}s")

incremental: (300, 5, 19), 0.079s


   Solution: (300, 5, 19), 0.072s


     binary: (300, 5, 19), 0.144s


  SolBinary: (300, 5, 19), 0.136s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Sai dấu trong công thức prefix (quên `+ P[i-k][j-k]`)

In [21]:
def wrong_formula(mat, threshold):
    m, n = len(mat), len(mat[0])
    P = build_prefix(mat)
    best = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            for k in range(1, min(i, j) + 1):
                s = P[i][j] - P[i - k][j] - P[i][j - k]          # SAI: thiếu + P[i-k][j-k]
                if s <= threshold:
                    best = max(best, k)
    return best

mat = [[1, 1, 1], [1, 1, 1], [1, 1, 1]]
print("Code sai:", wrong_formula(mat, 3), "| đúng là:", Solution().maxSideLength(mat, 3))

Code sai: 2 | đúng là: 1


Thiếu phần cộng lại, vùng chữ nhật phía trên-trái bị trừ **hai lần** ⇒ tổng nhỏ hơn thật: hình vuông 2×2 ở góc dưới-phải bị tính ra `9 − 3 − 3 = 3 ≤ 3` trong khi tổng thật là 4. (Hình vuông sát góc `(0,0)` không lộ lỗi vì phần bị thiếu bằng 0 — nên test phải có hình vuông ở giữa ma trận.) **Code đúng:** `P[i][j] - P[i-k][j] - P[i][j-k] + P[i-k][j-k]`.

### Lỗi 2 — Binary search "tìm lớn nhất" với `mid = (lo + hi) // 2` ⇒ lặp vô hạn

Thêm bộ đếm vòng để dừng an toàn và in ra.

In [22]:
def wrong_bs(mat, threshold, max_iter=100):
    m, n = len(mat), len(mat[0])
    P = build_prefix(mat)
    exists = lambda k: k == 0 or any(square_sum(P, i, j, k) <= threshold
                                     for i in range(k, m + 1) for j in range(k, n + 1))
    lo, hi, it = 0, min(m, n), 0
    while lo < hi:
        it += 1
        if it > max_iter:
            return f"kẹt ở lo={lo}, hi={hi} sau {max_iter} vòng"
        mid = (lo + hi) // 2                                     # SAI: làm tròn xuống
        if exists(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

print("Code sai:", wrong_bs(EX1, 4))
print("Code đúng:", Solution().maxSideLength(EX1, 4))

Code sai: kẹt ở lo=2, hi=3 sau 100 vòng
Code đúng: 2


Khi `hi = lo + 1` và `exists(lo)` đúng, `mid = lo` ⇒ `lo = mid` không đổi gì. **Code đúng:** `mid = (lo + hi + 1) // 2`.

### Lỗi 3 — Mẹo tăng dần nhưng quên kiểm tra `i ≥ k`, `j ≥ k` (chỉ số âm)

In [23]:
def wrong_negative_index(mat, threshold):
    m, n = len(mat), len(mat[0])
    P = build_prefix(mat)
    ans = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            k = ans + 1
            s = P[i][j] - P[i - k][j] - P[i][j - k] + P[i - k][j - k]   # SAI: i - k có thể âm
            if s <= threshold:
                ans = k
    return ans

mat = [[0, 9, 9], [9, 9, 9], [9, 9, 0]]
print("Code sai:", wrong_negative_index(mat, 0), "| đúng là:", Solution().maxSideLength(mat, 0))

Code sai: 5 | đúng là: 1


Ở hàng `i = 1`, thử `k = 2` ⇒ `P[-1]` là **hàng cuối** của `P` — Python không báo lỗi, tổng tính ra vô nghĩa ⇒ `ans` tăng bừa, thậm chí vượt cả kích thước ma trận. **Code đúng:** thêm điều kiện `i >= k and j >= k`.

### Lỗi 4 — Khởi tạo đáp án bằng 1

In [24]:
def wrong_init(mat, threshold):
    m, n = len(mat), len(mat[0])
    P = build_prefix(mat)
    ans = 1                                                   # SAI: có thể không có ô nào thoả
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            k = ans + 1
            if i >= k and j >= k and square_sum(P, i, j, k) <= threshold:
                ans = k
    return ans

print("Code sai:", wrong_init(EX2, 1), "| đúng là:", Solution().maxSideLength(EX2, 1))

Code sai: 1 | đúng là: 0


**Code đúng:** `ans = 0` — đề yêu cầu trả 0 khi không có hình vuông nào.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng của một **vùng chữ nhật/vuông**" lặp lại nhiều lần ⇒ **prefix sum 2D**.
- Phần tử **không âm** + "lớn nhất sao cho tổng ≤ ngưỡng" ⇒ **đơn điệu** ⇒ binary search trên đáp án, hoặc mẹo **tăng dần đáp án**.
- Mẹo `ans + 1`: khi đáp án chỉ có thể tăng và mỗi vị trí chỉ cần biết "có tốt hơn 1 bậc không" ⇒ giảm một chiều `log` hoặc `L`.
- Binary search tìm **lớn nhất**: `mid = (lo + hi + 1) // 2`.
- Luôn dùng bảng prefix có **viền 0** (kích thước `(m+1) × (n+1)`) để khỏi xử lý biên.

## 15. Biến thể của bài

1. **Đếm số hình vuông** có tổng `≤ threshold` (Bài tập 5) — mỗi ô binary search cạnh lớn nhất, cộng lại.
2. **Hình chữ nhật** thay vì hình vuông: LC 363 *Max Sum of Rectangle No Larger Than K* (khó hơn: không còn đơn điệu đơn giản, cần sorted list).
3. **Phần tử có thể âm:** mất tính đơn điệu ⇒ không binary search được; phải thử mọi cạnh `O(mn·L)`.
4. **LC 1277 Count Square Submatrices with All Ones**, **LC 221 Maximal Square** — họ bài hình vuông trong ma trận.
5. **LC 304 Range Sum Query 2D - Immutable** — bài nền tảng về prefix sum 2D.
6. **Tổng ≥ ngưỡng, cạnh nhỏ nhất:** đảo chiều đơn điệu, binary search "tìm nhỏ nhất".

In [25]:
class NumMatrix:                            # LC 304 — prefix sum 2D dạng lớp
    def __init__(self, matrix):
        self.P = build_prefix(matrix)
    def sumRegion(self, r1, c1, r2, c2):
        P = self.P
        return P[r2 + 1][c2 + 1] - P[r1][c2 + 1] - P[r2 + 1][c1] + P[r1][c1]

nm = NumMatrix([[3, 0, 1, 4, 2], [5, 6, 3, 2, 1], [1, 2, 0, 1, 5], [4, 1, 0, 1, 7], [1, 0, 3, 0, 5]])
print(nm.sumRegion(2, 1, 4, 3), nm.sumRegion(1, 1, 2, 2), nm.sumRegion(1, 2, 2, 4))   # 8 11 12

8 11 12


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử mọi hình vuông, cộng trực tiếp | `O(mn·L³)` | `O(1)` | kiểm chứng |
| 2. Prefix + thử mọi cạnh | tổng `O(1)`, `break` khi vượt | `O(mn·L)` | `O(mn)` | khi chưa thấy đơn điệu |
| 3. Prefix + binary search | `exists(k)` đơn điệu | `O(mn log L)` | `O(mn)` | dễ chứng minh, phổ biến |
| 4. Tăng dần `ans+1` | mỗi ô chỉ thử một cạnh | `O(mn)` | `O(mn)` | **nộp bài** |
| 5. BS từng ô | cạnh lớn nhất tại mỗi ô | `O(mn log L)` | `O(mn)` | biến thể đếm |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `maxSideLength([[1]], 0)`, `maxSideLength([[1]], 1)`, `maxSideLength([[0,0],[0,0]], 0)`?

**Bài 2 (tính tay):** Dựng bảng prefix `P` (4×4, có viền 0) cho `[[1,2,3],[4,5,6],[7,8,9]]` và tính đáp án với `threshold = 12` và `threshold = 28`.

**Bài 3 (điền TODO):** Hoàn thiện `exists_square_todo(P, m, n, k, threshold)`.

**Bài 4 (tự viết lại):** Tự viết lại Cách 4 nhưng tách riêng bước dựng `P` (không dựng đồng thời).

**Bài 5 (mini challenge):** Viết `count_squares_le(mat, threshold)` đếm **số hình vuông con** (mọi kích thước, mọi vị trí) có tổng `≤ threshold`, nhanh hơn `O(mn·L)` (gợi ý: `O(mn log L)`).

In [26]:
def exists_square_todo(P, m, n, k, threshold):
    # TODO: duyệt mọi góc dưới-phải (i, j) với k <= i <= m, k <= j <= n
    #       trả True nếu tổng hình vuông cạnh k <= threshold
    pass

def count_squares_le_todo(mat, threshold):
    P = build_prefix(mat)
    # TODO: với mỗi góc dưới-phải, tìm cạnh lớn nhất thoả bằng binary search và cộng vào đáp án
    pass

# Lưu ý: cell này chỉ định nghĩa hàm, không gọi.

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Một ô duy nhất có giá trị 1. (2) `1 > 0` ⇒ 0; `1 ≤ 1` ⇒ 1. (3) Toàn 0 với ngưỡng 0: cả hình vuông 2×2 có tổng 0.

**Bài 2.** (1) Hàng 1 của `P`: `0 1 3 6`; hàng 2: `0 5 12 21`; hàng 3: `0 12 27 45`. (2) Các 2×2: tổng `12, 16, 24, 28`. (3) `threshold = 12` ⇒ 2; `threshold = 28` ⇒ 2 (3×3 tổng 45 > 28).

**Bài 3.** (1) Hai vòng `for i in range(k, m+1)`, `for j in range(k, n+1)`. (2) Dùng công thức bao hàm–loại trừ. (3) `return True` ngay khi thấy; cuối cùng `return False`.

**Bài 4.** (1) `P = build_prefix(mat)`. (2) Duyệt `i, j` từ 1; `k = ans + 1`; kiểm tra biên. (3) So với brute force.

**Bài 5.** (1) Tại góc dưới-phải `(i, j)`, cạnh thoả có dạng `1..K(i,j)` (đơn điệu). (2) Số hình vuông thoả có góc này đúng bằng `K(i,j)`. (3) Tìm `K` bằng `last_true`, cộng tất cả.

## 19. Đáp án tham khảo

In [27]:
# Bài 1
assert Solution().maxSideLength([[1]], 0) == 0
assert Solution().maxSideLength([[1]], 1) == 1
assert Solution().maxSideLength([[0, 0], [0, 0]], 0) == 2

# Bài 2
P_small = build_prefix([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
assert P_small == [[0, 0, 0, 0], [0, 1, 3, 6], [0, 5, 12, 21], [0, 12, 27, 45]]
assert Solution().maxSideLength([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 12) == 2
assert Solution().maxSideLength([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 28) == 2

# Bài 3
def exists_square_sol(P, m, n, k, threshold):
    for i in range(k, m + 1):
        for j in range(k, n + 1):
            if P[i][j] - P[i - k][j] - P[i][j - k] + P[i - k][j - k] <= threshold:
                return True
    return False

# Bài 4
def max_side_incremental_sol(mat, threshold):
    m, n = len(mat), len(mat[0])
    P = build_prefix(mat)
    ans = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            k = ans + 1
            if i >= k and j >= k and square_sum(P, i, j, k) <= threshold:
                ans = k
    return ans

# Bài 5
def count_squares_le_sol(mat, threshold):
    m, n = len(mat), len(mat[0])
    P = build_prefix(mat)
    total = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            total += last_true(0, min(i, j), lambda k: k == 0 or square_sum(P, i, j, k) <= threshold)
    return total

def count_squares_brute(mat, threshold):
    m, n = len(mat), len(mat[0])
    return sum(1 for r in range(m) for c in range(n) for k in range(1, min(m - r, n - c) + 1)
               if sum(mat[i][j] for i in range(r, r + k) for j in range(c, c + k)) <= threshold)

rng = random.Random(31)
for _ in range(300):
    m, n = rng.randint(1, 7), rng.randint(1, 7)
    mat = [[rng.randint(0, 9) for _ in range(n)] for _ in range(m)]
    th = rng.randint(0, 60)
    P = build_prefix(mat)
    want = max_side_brute(mat, th)
    assert max_side_incremental_sol(mat, th) == want
    for k in range(1, min(m, n) + 1):
        assert exists_square_sol(P, m, n, k, th) == (k <= want)
    assert count_squares_le_sol(mat, th) == count_squares_brute(mat, th)
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template prefix sum 2D (viền 0):**

```python
P = [[0] * (n + 1) for _ in range(m + 1)]
for i in range(1, m + 1):
    for j in range(1, n + 1):
        P[i][j] = mat[i-1][j-1] + P[i-1][j] + P[i][j-1] - P[i-1][j-1]
# tổng hình vuông cạnh k, góc dưới-phải (i, j) của P:
s = P[i][j] - P[i-k][j] - P[i][j-k] + P[i-k][j-k]
```

**Template tăng dần đáp án:**

```python
ans = 0
for i in range(1, m + 1):
    for j in range(1, n + 1):
        k = ans + 1
        if i >= k and j >= k and sum_square(i, j, k) <= threshold:
            ans = k
```

**Checklist trước khi submit:**
- [ ] Công thức bao hàm–loại trừ đủ 4 số hạng, đúng dấu.
- [ ] Kiểm tra `i >= k` và `j >= k` (tránh chỉ số âm lặng lẽ).
- [ ] Binary search tìm lớn nhất dùng `mid = (lo + hi + 1) // 2`.
- [ ] `ans` khởi tạo 0; trả 0 khi không có hình vuông.
- [ ] Nhớ tính đơn điệu chỉ đúng vì `mat[i][j] ≥ 0`.
- [ ] Đúng chữ ký: `maxSideLength(self, mat: List[List[int]], threshold: int) -> int`.